# Import Library

In [24]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# Library Khusus Bahasa Indonesia
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import difflib
import re

# MLflow
import mlflow
import mlflow.sklearn
import dagshub

# Import Data

In [25]:
import pandas as pd

file_path = "../../data/synthetic_report_dataset.csv" # Pastikan nama/path file-nya benar

# 1. Kita intip dulu file mentahnya untuk melihat pemisahnya
with open(file_path, 'r', encoding='utf-8') as f:
    baris_pertama = f.readline()
    print("Isi baris pertama file mentah:\n👉", baris_pertama)

# 2. Auto-Deteksi Pemisah Kolom
if '|' in baris_pertama:
    pemisah = '|'
else:
    pemisah = ','
print(f"Sistem mendeteksi pemisah kolom: '{pemisah}'\n")

# 3. Load Data
# Menggunakan on_bad_lines='skip' hanya untuk baris yang benar-benar hancur (misal kelebihan koma)
nama_kolom = ['teks_keluhan_awam', 'teks_laporan_teknisi', 'kategori_aset', 'severity', 'root_cause', 'tindakan']
df = pd.read_csv(file_path, sep=pemisah, names=nama_kolom, on_bad_lines='skip')

print(f"Baris sebelum dibersihkan: {df.shape[0]}")

# 4. PEMBERSIHAN & PENYELAMATAN DATA (Pandas Recovery Magic)
kolom_target = ['kategori_aset', 'severity', 'root_cause', 'tindakan']

# a. Deteksi baris patahan (baris yang kepotong enter biasanya kolom targetnya NaN)
baris_patah = df[df['kategori_aset'].isna()].index

# b. Selamatkan teks yang terpotong!
for i in baris_patah:
    # Cek apakah baris di bawahnya ada dan targetnya tidak kosong (berarti itu lanjutannya)
    if i + 1 in df.index and not pd.isna(df.loc[i+1, 'kategori_aset']):
        teks_atas = str(df.loc[i, 'teks_keluhan_awam']).strip()
        teks_bawah = str(df.loc[i+1, 'teks_keluhan_awam']).strip()
        
        # Tempelkan kembali potongan teks ke baris bawahnya
        if teks_atas != 'nan':
            df.loc[i+1, 'teks_keluhan_awam'] = teks_atas + " " + teks_bawah

# c. Setelah teks diselamatkan, baru kita buang baris patahan / NaN tersebut
df = df.dropna(subset=kolom_target)

# d. Buang header CSV asli jika ikut terbaca ke dalam baris data
df = df[df['teks_keluhan_awam'].astype(str).str.contains('teks_keluhan', case=False) == False]

# Reset nomor urut baris agar berurutan kembali
df = df.reset_index(drop=True)

# Pastikan jadi string agar tidak error saat digabung
df['teks_keluhan_awam'] = df['teks_keluhan_awam'].astype(str)
df['teks_laporan_teknisi'] = df['teks_laporan_teknisi'].astype(str)

# 5. Gabungkan Teks (Sebagai Input X)
df['input_teks'] = df['teks_keluhan_awam'] + " " + df['teks_laporan_teknisi']

# 6. MENGAMBIL KOLOM TARGET (Sebagai Kunci Jawaban Y)
Y = df[kolom_target] 

print(f"Total data SIAP DITRAINING: {df.shape[0]}")
print(f"Bentuk Input (X): 1 Kolom Teks Gabungan")
print(f"Bentuk Target (Y): {Y.shape[1]} Kolom Kunci Jawaban")
print("\n--- Intip Isi Target (Y) ---")
display(Y.head(3))

Isi baris pertama file mentah:
👉 AC di ruang meeting lantai 2 netes air terus nih,"Saluran pembuangan air kondensasi tersumbat lumut dan debu, sudah dibersihkan tuntas",HVAC,Sedang,Tersumbat,Pembersihan

Sistem mendeteksi pemisah kolom: ','

Baris sebelum dibersihkan: 2797
Total data SIAP DITRAINING: 2796
Bentuk Input (X): 1 Kolom Teks Gabungan
Bentuk Target (Y): 4 Kolom Kunci Jawaban

--- Intip Isi Target (Y) ---


,kategori_aset,severity,root_cause,tindakan
0,HVAC,Sedang,Tersumbat,Pembersihan
1,Pompa Air,Tinggi,Keausan,Penggantian Part
2,Mesin Produksi,Tinggi,Tersumbat,Pembersihan


In [26]:
import pandas as pd

# 1. CARA TERBAIK: Cek Distribusi / Imbalance Kelas
print("=== Distribusi Label di Kolom Severity ===")
# dropna=False agar kita bisa lihat kalau tiba-tiba ada yang kosong (NaN)
print(Y['severity'].value_counts(dropna=False)) 

print("\n-------------------------------------------------\n")

# 2. CARA CEK TYPO: Melihat semua nilai unik yang pernah diketik
print("=== Nilai Unik di Kolom Severity ===")
print(Y['severity'].unique())

=== Distribusi Label di Kolom Severity ===
severity
Tinggi     1173
Sedang     1011
Rendah      611
Rendang       1
Name: count, dtype: int64

-------------------------------------------------

=== Nilai Unik di Kolom Severity ===
['Sedang' 'Tinggi' 'Rendah' 'Rendang']


# Cleaning & Preprocessing

In [27]:
# ==============================================================
# TAHAP 1: BERSIHKAN TYPO DI LABEL TARGET (KUNCI JAWABAN Y)
# ==============================================================
print("1. Memperbaiki Typo pada Label Target...")
valid_severity = ['Tinggi', 'Sedang', 'Rendah']

def auto_fix_typo(text, valid_list):
    text = str(text).strip()
    closest_match = difflib.get_close_matches(text, valid_list, n=1, cutoff=0.6)
    return closest_match[0] if closest_match else text

# Terapkan HANYA ke kolom target
df['severity'] = df['severity'].apply(lambda x: auto_fix_typo(x, valid_severity))

# (Lakukan hal yang sama untuk kategori_aset, dll jika perlu)


# ==============================================================
# TAHAP 2: PREPROCESSING NLP PADA INPUT TEKS (FITUR X)
# ==============================================================
print("2. Sedang melakukan Preprocessing teks masukan (Tunggu sebentar)...")
stemmer = StemmerFactory().create_stemmer()
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()

def clean_preprocessing(text):
    # a. Case Folding (Lowercase)
    text = text.lower()
    # b. Cleaning (Hapus karakter spesial & angka)
    text = re.sub(r'[^a-z\s]', '', text)
    # c. Stopword Removal
    text = stopword_remover.remove(text)
    # d. Stemming (Mengubah ke kata dasar)
    text = stemmer.stem(text)
    return text

# Terapkan HANYA ke kolom teks keluhan gabungan
df['clean_teks'] = df['input_teks'].apply(clean_preprocessing)

print("Data Cleaning Selesai!")

1. Memperbaiki Typo pada Label Target...
2. Sedang melakukan Preprocessing teks masukan (Tunggu sebentar)...
Data Cleaning Selesai!


# Train-Test Split

In [28]:
# Pisahkan sebelum Augmentasi untuk mencegah Data Leakage!
X = df['clean_teks']
y = df[kolom_target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# DATA AUGMENTATION

In [29]:
def synonym_augmentation(text):
    # Contoh sederhana Synonym Replacement
    synonyms = {"panas": "overheat", "rusak": "kendala", "bocor": "netes", "mati": "padam"}
    words = text.split()
    augmented_words = [synonyms.get(w, w) for w in words]
    return " ".join(augmented_words)

# Menambah jumlah data latih 2x lipat dengan augmentasi
X_train_aug = X_train.apply(synonym_augmentation)
X_train_final = pd.concat([X_train, X_train_aug])
y_train_final = pd.concat([y_train, y_train])

print(f"Data setelah Augmentasi: {X_train_final.shape[0]} baris.")

Data setelah Augmentasi: 4472 baris.


# FEATURE ENGINEERING & MODELING

In [30]:
# --- SETUP MLFLOW & DAGSHUB ---
# 1. Inisialisasi DagsHub (Otomatis mengatur tracking URI ke cloud)
# Hapus atau comment mlflow.set_tracking_uri lokal
# mlflow.set_tracking_uri("file:../../mlruns") 

dagshub.init(repo_owner='NazeeraAlthea', repo_name='exigen-smart-maintenance', mlflow=True)

# 2. Buat atau pilih eksperimen
mlflow.set_experiment("Smart_Ticketing_Baseline_TFIDF")

# --- MEMULAI TRACKING ---
# Dengan 'with', sesi MLflow akan otomatis ditutup setelah kode selesai dieksekusi
with mlflow.start_run(run_name="TFIDF_RF_Hyperparameter_Tuning"):
    
    print("Mulai Hyperparameter Tuning dengan GridSearchCV dan MLflow Tracking...")
    
    # 1. Mendefinisikan Pipeline dan Grid
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer()),
        ('clf', MultiOutputClassifier(RandomForestClassifier(random_state=42)))
    ])

    param_grid = {
        'tfidf__max_features': [1000, 2000],
        'clf__estimator__n_estimators': [100, 200],
        'clf__estimator__max_depth': [None, 10, 20]
    }

    # 2. Proses Latihan (Training)
    grid_search = GridSearchCV(pipeline, param_grid, cv=3, n_jobs=1, verbose=1)
    grid_search.fit(X_train_final, y_train_final)

    print(f"\nBest Parameters: {grid_search.best_params_}")
    
    # --- LOGGING KE MLFLOW (DAGSHUB) ---
    # a. Catat Parameter Terbaik
    mlflow.log_params(grid_search.best_params_)
    
    # b. Evaluasi & Catat Metrik
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    print("\n=== HASIL EVALUASI FINAL ===")
    
    # Hitung & Catat Exact Match
    exact_match_manual = (y_test.values == y_pred).all(axis=1).mean()
    mlflow.log_metric("exact_match_ratio", exact_match_manual)
    print(f"Exact Match Ratio (Benar Semua 4 Kolom): {exact_match_manual * 100:.2f}%")

    print("\n--- Akurasi Individu per Target ---")
    for i, col in enumerate(kolom_target):
        acc = accuracy_score(y_test.iloc[:, i], y_pred[:, i])
        # Catat metrik per target/kolom ke MLflow
        mlflow.log_metric(f"accuracy_{col}", acc)
        print(f"Akurasi {col:15}: {acc * 100:.2f}%")
        
    # c. Simpan (Log) Model Terbaik
    mlflow.sklearn.log_model(best_model, "best_rf_tfidf_model")
    print("\n✅ Run MLflow Selesai! Model dan metrik berhasil dikirim ke DagsHub.")

Initialized MLflow to track repo "NazeeraAlthea/exigen-smart-maintenance"

Repository NazeeraAlthea/exigen-smart-maintenance initialized!

Mulai Hyperparameter Tuning dengan GridSearchCV dan MLflow Tracking...
Fitting 3 folds for each of 12 candidates, totalling 36 fits

Best Parameters: {'clf__estimator__max_depth': None, 'clf__estimator__n_estimators': 200, 'tfidf__max_features': 1000}

=== HASIL EVALUASI FINAL ===
Exact Match Ratio (Benar Semua 4 Kolom): 78.04%

--- Akurasi Individu per Target ---
Akurasi kategori_aset  : 100.00%
Akurasi severity       : 78.57%
Akurasi root_cause     : 99.29%


2026/05/10 00:24:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Akurasi tindakan       : 99.29%


2026/05/10 00:24:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



✅ Run MLflow Selesai! Model dan metrik berhasil dikirim ke DagsHub.
🏃 View run TFIDF_RF_Hyperparameter_Tuning at: https://dagshub.com/NazeeraAlthea/exigen-smart-maintenance.mlflow/#/experiments/1/runs/e2671825b938460caf629cf245b53977
🧪 View experiment at: https://dagshub.com/NazeeraAlthea/exigen-smart-maintenance.mlflow/#/experiments/1
